# Using the Webcam

Accessing your device's webcam is easy in Codetto. Once you turn it on you can show the live video, flip it like a mirror, change its size, and grab still pictures from it — all in a few lines of code.

Everything runs **on your own computer** within the browser. No video is uploaded anywhere.

The first time you start the camera your browser will ask for permission. Click **Allow**. If you accidentally block it, look for a small camera icon near the address bar to change your mind.

Run the cell below to switch the camera on for five seconds.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)

time.sleep(5)

camera.stop()

# The Camera and the Canvas

Two things work together:

- `graphics.canvas()` makes a **canvas** in the notebook for the picture to appear in. With nothing in the brackets it sizes itself to fit the notebook's cell.
- `cv.start_camera(canvas)` turns the webcam on and starts drawing each video frame onto that canvas.

`start_camera` hands you back a **camera object**. Keep it in a variable (here it is called `camera`) so you can control it later. The most important thing you can do with it is `camera.stop()`, which switches the webcam off.

Always stop a camera when you are finished with it, otherwise the camera light stays on until you re-run the cell.

# Keeping the Camera On

In the first example the camera ran for exactly five seconds because of `time.sleep(5)`. When that line finished, so did the cell.

Usually you want the video to stay on until *you* decide to stop it. For that, we can keep the cell busy with a loop and end it with the **Stop** button (■ in the toolbar):

- `while True:` repeats forever.
- `time.sleep(0.1)` inside the loop keeps it from racing.
- Putting `camera.stop()` in a `finally:` block means the camera still switches off cleanly when you press Stop.

Run the cell, watch yourself for a while, then press **Stop**.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)

try:
  while True:
    time.sleep(0.1)
finally:
  camera.stop()

# Mirror Mode

By default the camera shows exactly what it sees. This can feel odd if you are looking at yourself. Move your hand to the right and it goes left on screen.

Pass `mirror=True` to flip the video left-to-right, the way a real mirror or a video call does. Now your movements match the picture. Any writing behind you will read backwards, which is a quick way to tell that mirroring is on.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas, mirror=True)

time.sleep(5)

camera.stop()

# Resizing the Video

Give `graphics.canvas()` a **width** and a **height** in pixels to fix how big the video is. For example, `graphics.canvas(320, 240)`.

The webcam picture is stretched or squashed to fill whatever canvas you give it. A canvas shaped `320×240` (a wide 4:3 rectangle) keeps things looking natural, while `300×300` would make everything look a little tall and narrow.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas(320, 240)
camera = cv.start_camera(canvas)

time.sleep(5)

camera.stop()

# Taking a Picture

`cv.capture_frame(camera)` grabs a single still frame from the running camera and gives it back to you. `graphics.display_image(...)` then shows that picture in the output.

To make it feel like a real photo booth, the cell below draws a **3… 2… 1…** countdown on a see-through layer over the video, then takes the photo.

`canvas.get_context('2d')` gives you a drawing layer that sits *on top of* the webcam video and is not painted over by it, so anything you draw there stays put. `canvas.clear()` wipes that layer without touching the video. The countdown never ends up in the saved photo — `capture_frame` copies only the camera image, not what you draw on top.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas, mirror=True)
ctx = canvas.get_context('2d')

for n in (3, 2, 1):
  canvas.clear()
  ctx.fill_style = 'rgba(0, 0, 0, 0.45)'
  ctx.fill_rect(0, 0, canvas.get_width(), canvas.get_height())
  ctx.fill_style = '#ffffff'
  ctx.font = f'{int(canvas.get_height() * 0.5)}px sans-serif'
  ctx.text_align = 'center'
  ctx.text_baseline = 'middle'
  ctx.fill_text(str(n), canvas.get_width() / 2, canvas.get_height() / 2)
  time.sleep(1)

canvas.clear()
photo = cv.capture_frame(camera)
camera.stop()

graphics.display_image(photo)

# What You Get Back

`capture_frame` does not return a picture object you can draw on, it returns a **string**. The string is known as a *data URL*: the whole image is packed into text that starts with `data:image/jpeg;base64,` followed by thousands of letters and numbers.

That format is useful because you can pass it straight to other tools. This could be `graphics.display_image()` as we used here, or an AI image model in a more advanced project. Run the cell to peek at the start of the data from the photo you just took.

In [ ]:
print(type(photo))
print(photo[:40], '...')
print('Length:', len(photo), 'characters')

# Check Your Understanding